In [ ]:
# Nomic BERT 2048 Interactive Notebook
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig
import numpy as np
from typing import Optional, List
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3. Load the Nomic BERT Model

# Model identifier on Hugging Face
model_name = "AbstractPhil/bert-beatrix-2048"

# Load configuration
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
print(f"Model config loaded. Max position embeddings: {config.max_position_embeddings}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Load model
model = AutoModel.from_pretrained(
    model_name,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)

print(f"Model loaded successfully!")
print(f"Model type: {type(model)}")

def encode_text(text: str, return_tensors: bool = True):
    """
    Encode text into tokens and optionally return tensors.
    """
    encoded = tokenizer(
        text,
        return_tensors='pt' if return_tensors else None,
        padding=True,
        truncation=True,
        max_length=2048
    )

    if return_tensors:
        return {k: v.to(device) for k, v in encoded.items()}
    return encoded

def get_embeddings(text: str, pool_type: str = 'cls') -> torch.Tensor:
    """
    Get embeddings for input text.

    Args:
        text: Input text string
        pool_type: Type of pooling ('cls', 'mean', 'max')

    Returns:
        Embeddings tensor
    """
    # Encode text
    inputs = encode_text(text)

    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)

    # Get embeddings based on pooling type
    if pool_type == 'cls':
        # Use CLS token (first token) embedding
        embeddings = outputs.last_hidden_state[:, 0, :]
    elif pool_type == 'mean':
        # Mean pooling
        attention_mask = inputs['attention_mask']
        embeddings = (outputs.last_hidden_state * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1).unsqueeze(-1)
    elif pool_type == 'max':
        # Max pooling
        embeddings = outputs.last_hidden_state.max(dim=1)[0]
    else:
        raise ValueError(f"Unknown pool_type: {pool_type}")

    return embeddings.cpu()

def compute_similarity(text1: str, text2: str, pool_type: str = 'cls') -> float:
    """
    Compute cosine similarity between two texts.
    """
    # Get embeddings
    emb1 = get_embeddings(text1, pool_type)
    emb2 = get_embeddings(text2, pool_type)

    # Compute cosine similarity
    similarity = torch.nn.functional.cosine_similarity(emb1, emb2, dim=1)

    return similarity.item()

def interactive_chat():
    """
    Interactive function to chat with BERT about its embeddings.
    """
    print("BERT Embedding Explorer")
    print("=" * 50)
    print("Enter texts to get embeddings and compute similarities.")
    print("Type 'quit' to exit.")
    print("=" * 50)

    texts = []

    while True:
        text = input("\nEnter text (or 'quit' to exit, 'compare' to compare texts): ")

        if text.lower() == 'quit':
            break
        elif text.lower() == 'compare' and len(texts) >= 2:
            print("\nComparing last two texts:")
            sim = compute_similarity(texts[-2], texts[-1])
            print(f"Text 1: {texts[-2][:50]}...")
            print(f"Text 2: {texts[-1][:50]}...")
            print(f"Cosine similarity: {sim:.4f}")
        else:
            texts.append(text)
            emb = get_embeddings(text)
            print(f"Embedding shape: {emb.shape}")
            print(f"Embedding norm: {torch.norm(emb).item():.4f}")
            print(f"First 5 values: {emb[0][:5].numpy()}")


def analyze_model():
    """
    Analyze the model architecture and capabilities.
    """
    print("Model Architecture Analysis")
    print("=" * 50)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Check model configuration
    print(f"\nModel Configuration:")
    print(f"- Hidden size: {config.hidden_size}")
    print(f"- Number of layers: {config.num_hidden_layers}")
    print(f"- Number of attention heads: {config.num_attention_heads}")
    print(f"- Max position embeddings: {config.max_position_embeddings}")
    print(f"- Vocabulary size: {config.vocab_size}")

    # Check special features
    if hasattr(config, 'rotary_emb_fraction'):
        print(f"- Rotary embedding fraction: {config.rotary_emb_fraction}")
    if hasattr(config, 'use_flash_attn'):
        print(f"- Flash attention: {config.use_flash_attn}")
    if hasattr(config, 'moe_num_experts'):
        print(f"- MoE experts: {config.moe_num_experts}")

    return config

## Run analysis
#config_info = analyze_model()
#```
#
### 6. Example Usage: Text Embeddings
#
#```python
## Example texts
#texts = [
#    "The quick brown fox jumps over the lazy dog.",
#    "A fast auburn fox leaps above a sleepy canine.",
#    "Machine learning is transforming the world.",
#    "Artificial intelligence is changing our planet.",
#    "The weather is nice today."
#]
#
## Compute embeddings for all texts
#print("Computing embeddings for example texts...")
#embeddings = [get_embeddings(text) for text in texts]
#
## Create similarity matrix
#similarity_matrix = torch.zeros(len(texts), len(texts))
#for i in range(len(texts)):
#    for j in range(len(texts)):
#        if i != j:
#            similarity_matrix[i, j] = compute_similarity(texts[i], texts[j])
#        else:
#            similarity_matrix[i, j] = 1.0
#
## Display results
#print("\nSimilarity Matrix:")
#print("-" * 80)
#for i, text in enumerate(texts):
#    print(f"{i}: {text[:40]}...")
#
#print("\n   ", end="")
#for i in range(len(texts)):
#    print(f"  {i}   ", end="")
#print()
#
#for i in range(len(texts)):
#    print(f"{i}: ", end="")
#    for j in range(len(texts)):
#        print(f"{similarity_matrix[i, j]:.3f} ", end="")
#    print()
#```
#
### 7. Long Text Handling Demo
#
#```python
#def test_long_text_handling():
#    """
#    Test how the model handles long texts (up to 2048 tokens).
#    """
#    print("\nTesting long text handling...")
#    print("=" * 50)
#
#    # Create texts of different lengths
#    base_sentence = "This is a test sentence. "
#
#    test_lengths = [10, 50, 100, 500, 1000, 2000]
#
#    for length in test_lengths:
#        # Create text with approximately 'length' tokens
#        num_repeats = length // 5  # Approximate tokens per sentence
#        text = base_sentence * num_repeats
#
#        # Tokenize
#        tokens = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048)
#        num_tokens = tokens['input_ids'].shape[1]
#
#        # Get embeddings
#        inputs = {k: v.to(device) for k, v in tokens.items()}
#        with torch.no_grad():
#            outputs = model(**inputs)
#
#        print(f"Target length: {length} tokens")
#        print(f"Actual tokens: {num_tokens}")
#        print(f"Output shape: {outputs.last_hidden_state.shape}")
#        print("-" * 40)
#
#test_long_text_handling()
#```
#
### 8. Interactive Embedding Explorer
#
#```python

In [ ]:
def embedding_explorer():
    """
    Interactive tool to explore embeddings and similarities.
    """
    print("\n🤖 Nomic BERT 2048 Interactive Explorer")
    print("=" * 50)
    print("Commands:")
    print("  add <text>     - Add a text to the collection")
    print("  compare i j    - Compare texts at indices i and j")
    print("  show           - Show all texts")
    print("  matrix         - Show similarity matrix")
    print("  clear          - Clear all texts")
    print("  quit           - Exit")
    print("=" * 50)

    texts = []

    while True:
        command = input("\n> ").strip().lower()

        if command == 'quit':
            print("Goodbye!")
            break

        elif command.startswith('add '):
            text = command[4:]
            texts.append(text)
            emb = get_embeddings(text)
            print(f"Added text #{len(texts)-1}: '{text[:50]}...'")
            print(f"Embedding shape: {emb.shape}")

        elif command.startswith('compare '):
            try:
                parts = command.split()
                i, j = int(parts[1]), int(parts[2])
                if 0 <= i < len(texts) and 0 <= j < len(texts):
                    sim = compute_similarity(texts[i], texts[j])
                    print(f"Similarity between texts {i} and {j}: {sim:.4f}")
                else:
                    print("Invalid indices!")
            except:
                print("Usage: compare i j")

        elif command == 'show':
            if not texts:
                print("No texts added yet!")
            else:
                for i, text in enumerate(texts):
                    print(f"{i}: {text[:60]}...")

        elif command == 'matrix':
            if len(texts) < 2:
                print("Need at least 2 texts for similarity matrix!")
            else:
                print("\nSimilarity Matrix:")
                for i in range(len(texts)):
                    for j in range(len(texts)):
                        if i == j:
                            print("1.00 ", end="")
                        else:
                            sim = compute_similarity(texts[i], texts[j])
                            print(f"{sim:.2f} ", end="")
                    print()

        elif command == 'clear':
            texts = []
            print("All texts cleared!")

        else:
            print("Unknown command. Type 'quit' to exit.")

# Uncomment to run the interactive explorer
# embedding_explorer()
#```
#
### 9. Batch Processing Example
#
#```python
def batch_embeddings(texts: List[str], batch_size: int = 8):
    """
    Process multiple texts in batches for efficiency.
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # Tokenize batch
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=2048
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            # Use CLS token embeddings
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu()
            all_embeddings.append(batch_embeddings)

    return torch.cat(all_embeddings, dim=0)

## Example usage
#sample_texts = [f"This is sample text number {i}." for i in range(20)]
#embeddings = batch_embeddings(sample_texts)
#print(f"Processed {len(sample_texts)} texts")
#print(f"Embeddings shape: {embeddings.shape}")
#```

## 10. Save and Load Embeddings

def save_embeddings(embeddings: torch.Tensor, texts: List[str], filename: str):
    """
    Save embeddings and associated texts.
    """
    torch.save({
        'embeddings': embeddings,
        'texts': texts,
        'model_name': model_name,
        'embedding_dim': embeddings.shape[1]
    }, filename)
    print(f"Saved {len(texts)} embeddings to {filename}")

def load_embeddings(filename: str):
    """
    Load embeddings and associated texts.
    """
    data = torch.load(filename)
    return data['embeddings'], data['texts']

# Example
# save_embeddings(embeddings, sample_texts, 'nomic_embeddings.pt')
# loaded_emb, loaded_texts = load_embeddings('nomic_embeddings.pt')


## Quick Start Guide

#Run these cells in order:
#1. Install dependencies (Cell 1)
#2. Import libraries and check device (Cell 2)
#3. Load the model (Cell 3)
#4. Run the analysis to understand the model (Cell 5)
#5. Try the example usage (Cell 6)
#6. Explore with the interactive functions!
#The model supports sequences up to 2048 tokens and provides high-quality embeddings for semantic similarity tasks.


In [1]:
"""
merge_gptoss_lora_robust.py

Merges a PEFT LoRA adapter into openai/gpt-oss-20b with minimal CUDA requirements.
Designed for a 24GB 4090: uses CPU/disk offload to avoid OOMs and avoids Flash/Triton paths.
"""

import os
import json
import torch
from pathlib import Path

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import PeftModel

# ---------- USER SETTINGS ----------
BASE_MODEL = "openai/gpt-oss-20b"   # must match what you trained against
ADAPTER_DIR = r"G:\My Drive\mirel\adapter"  # contains adapter_model.safetensors + adapter_config.json
OUT_DIR     = r"G:\My Drive\mirel\merged_gptoss"
OFFLOAD_DIR = r"G:\My Drive\mirel\offload_cache"  # disk offload cache
DTYPE       = torch.bfloat16  # gpt-oss supports bf16; use float16 if you must
# -----------------------------------

def env_sanity():
    print("== Environment ==")
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device:", torch.cuda.get_device_name(0))
        print("capability:", torch.cuda.get_device_capability(0))
    # Avoid Flash2/Triton paths
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
    os.environ.setdefault("BITSANDBYTES_NOWELCOME", "1")
    # Force eager attention to dodge flash/triton requirements
    os.environ.setdefault("HF_ATTENTION_BACKEND", "eager")

def load_tokenizer(base):
    print("Loading tokenizer…")
    tok = AutoTokenizer.from_pretrained(
        base,
        use_fast=True,
        trust_remote_code=True,  # gpt-oss uses custom code paths
    )
    return tok

def load_base_model_cpu_offload(base, dtype, offload_dir):
    """
    Safe path that works without flash-attn/triton.
    Loads the full model on CPU, with state_dict offload to disk while instantiating.
    """
    print("Loading base (CPU + disk offload)…")
    Path(offload_dir).mkdir(parents=True, exist_ok=True)
    model = AutoModelForCausalLM.from_pretrained(
        base,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        device_map={"": "cpu"},            # force CPU
        offload_folder=offload_dir,        # stream weights
        offload_state_dict=True,
        trust_remote_code=True,
        attn_implementation="eager",       # avoid flash requirements
    )
    return model

def attach_lora_and_merge(model, adapter_dir):
    print("Attaching LoRA…")
    peft = PeftModel.from_pretrained(
        model,
        adapter_dir,
        is_trainable=False,
    )
    print("Merging LoRA → base (this may take a while)…")
    # safe_merge keeps numerics reasonable; progressbar for visibility
    merged = peft.merge_and_unload(safe_merge=True, progressbar=True)
    return merged

def save_merged(model, tok, out_dir):
    print("Saving merged model…")
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    # Use safetensors (will shard automatically if large)
    model.save_pretrained(out_dir, safe_serialization=True, max_shard_size="4GB")
    tok.save_pretrained(out_dir)
    # Quick note file
    with open(Path(out_dir, "MERGE_NOTES.txt"), "w", encoding="utf-8") as f:
        f.write("Merged gpt-oss-20b + LoRA via peft.merge_and_unload (bf16, CPU/disk offload).\n")
    print(f"Done → {out_dir}")

def main():
    env_sanity()
    tok = load_tokenizer(BASE_MODEL)
    base = load_base_model_cpu_offload(BASE_MODEL, DTYPE, OFFLOAD_DIR)
    merged = attach_lora_and_merge(base, ADAPTER_DIR)
    # Free the PEFT wrapper early
    del base
    torch.cuda.empty_cache()
    save_merged(merged, tok, OUT_DIR)

if __name__ == "__main__":
    main()


I:\AIImageGen\ComfyUI\comfyui_windows_portable_sd3\new_ComfyUI_portable\ComfyUI\python_embeded\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


== Environment ==
torch: 2.5.1+cu121
cuda available: True
cuda device: NVIDIA GeForce RTX 4090
capability: (8, 9)
Loading tokenizer…
Loading base (CPU + disk offload)…


ValueError: The checkpoint you are trying to load has model type `gpt_oss` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`